In [ ]:
import torch
from src.model_unet import UNet,UnetGenerator
from src.DWM.data_setup import get_test_loader,get_watermark_img
from src.DWM.engine import *
from src.DWM.deep_watermark import *
import os

save_result = rf'runs2\watermark/lena_256_rgb_3'
print(save_result.split('/'))
name = save_result.split('/')
name = name[-1]
folder_name = rf'L1Unstructuredpuring2\{name}'
os.makedirs(folder_name,exist_ok=True)
p = 0.5
wm_size = (256,256)
round_wm = False
wm_channel = 3
watermark_acc = calculate_bcr
watermark_loss = nn.BCELoss()
adversarial_loss = nn.BCELoss()
task_loss = nn.L1Loss()
img_wm = get_watermark_img(rf'datasets\watermark_img\Spring-flowers-clipart-free-clipart-images.png',
                           wm_size,
                           convert='RGB',
                           round=round_wm)
val_path = rf'datasets\new_noise_dataset2\test'
dataloader = get_test_loader(val_path,
                            batch_size=16,
                            imgsize=(256,256),
                            convert='RGB')



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sk = torch.ones((1,)+wm_size)

model_H = UnetGenerator(input_nc=3,output_nc=3).to(device)
model_D = D_Watermark().to(device)
model_E = Extraction(wh=wm_size,
                        input_Eb=1,
                        output_Eb=1,
                        WH=(256,256),
                        input_Ea=3,
                        output_nc=wm_channel,
                        input_inception=8).to(device)





In [ ]:
def accuracy_fn(y_pred,Y):
    temp = torch.round(y_pred)
    acc = 1 - torch.mean(input=temp.to(torch.int) ^ Y.to(torch.int),dtype=torch.float)
    
    return  acc
def test_wm(test_dataloader,
            img_wm,
            secret_key,
            model_: torch.nn.Module,
            model_E: torch.nn.Module,
            content_loss,
            task_loss,
            accuracy_fn,
            calculate_watermark,
            device,save_name):

    wloss_list,tloss_list = [],[]
    wacc_list,tacc_list = [],[]
    # put model in eval mode
    model_.eval() 
    model_E.eval()

    # Turn on inference context manager
    with torch.inference_mode(): 

        val_desc = f'test:'
        with tqdm(test_dataloader, desc=val_desc,
                  bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}') as test_tqdm:

            for i,(input_img, target_img) in enumerate(test_tqdm):
            
                watermark_batch = img_wm.unsqueeze(0).repeat(input_img.size(0), 1, 1, 1)
                secret_key_batch = secret_key.unsqueeze(0).repeat(input_img.size(0), 1, 1, 1)
                
                input_img, target_img = input_img.to(device), target_img.to(device)
                watermark_batch,secret_key_batch = watermark_batch.to(device),secret_key_batch.to(device)
                
                wm_zero = torch.rand_like(watermark_batch).to(device)
                secret_x = torch.rand_like(secret_key_batch).to(device)

               
                # 1. Forward pass
                # input_h = torch.cat([input_img,watermark_batch],dim=1)
                
                generated_imgs = model_(input_img)
                
                # 2. Calculate loss and accuracy
                g_task = task_loss(generated_imgs, target_img)
                tloss_list.append(g_task.cpu().item())
                acc_task = accuracy_fn(generated_imgs, target_img)
                tacc_list.append(acc_task.cpu().item())

                ## Wloss1 = l1loss(E(H(x),k),w)
                watermark_extract = model_E(generated_imgs,secret_key_batch)
                w_loss1 = content_loss(watermark_extract,watermark_batch)
                ## Wloss2 = l1loss(E(x,k),wz)
                w_loss2 = content_loss(model_E(target_img,secret_key_batch),wm_zero)
                ## Wloss3 = l1loss(E(H(x),kx),wz)
                w_loss3 = content_loss(model_E(generated_imgs,secret_x),wm_zero)

                w_loss = w_loss1 + w_loss2 + 0.5*w_loss3
                wloss_list.append(w_loss.cpu().item())
                w_acc = calculate_watermark(watermark_batch,watermark_extract)
                wacc_list.append(w_acc.cpu().item())
                
                test_tqdm.set_description(
                    f'{val_desc} tloss:{np.mean(tloss_list):.3f} tacc:{np.mean(tacc_list):.3f} wloss:{np.mean(wloss_list):.3f} wacc:{np.mean(wacc_list):.3f}')

                if i == 0:

                    plt.imshow(np.transpose(generated_imgs[0].detach().cpu().numpy(),(1,2,0)))
                    plt.axis('off')
                    plt.savefig(rf'{folder_name}/{save_name}_output.png',bbox_inches='tight')
                    plt.show()

                    if watermark_extract.shape[1] == 1:
                        watermark_extract = watermark_extract[0,0].detach().cpu().numpy()
                        if round_wm:
                            watermark_extract = np.round(watermark_extract)
                        plt.imshow(watermark_extract,cmap='gray')
                    else:
                        plt.imshow(np.transpose(watermark_extract[0].detach().cpu().numpy(),(1,2,0)))
                
                    plt.axis('off')
                    plt.savefig(rf'{folder_name}/{save_name}_watermark.png',bbox_inches='tight')
                    plt.show()
    



                    
                
    # Calculate loss and accuracy per epoch and print out what's happening
    return np.mean(tloss_list),np.mean(tacc_list),np.mean(wloss_list),np.mean(wacc_list)

In [ ]:
def load_model(path,model):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['state_dict'])

    return model

In [ ]:
import torch.nn.utils.prune as prune
def pure_model(model,amount):
    

    # اعمال هرس به لایه‌های شناسایی شده
    parameters_to_prune = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d) or isinstance(module, nn.ConvTranspose2d):
            parameters_to_prune.append((module, 'weight'))


    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount,
    )
            
    # 3. حذف ماسک‌ها و اعمال هرس دائم
    for module, param_name in parameters_to_prune:
        prune.remove(module, param_name)


    return model


import torch.nn.utils.prune as prune
import torch.nn as nn



#### load model

In [ ]:
model_H = load_model(save_result+'/H/best.pt',model_H)
model_E = load_model(save_result+'/E/best.pt',model_E)


In [ ]:

#### puring model_H

model_P = pure_model(model_H,amount=p)
atloss,atacc,awloss,awacc = test_wm(dataloader,img_wm,sk,model_P,model_E,watermark_loss,task_loss,calculate_psnr,watermark_acc,device,'after')


In [ ]:
import pandas as pd 

df = pd.DataFrame({'before':[btloss,btacc,bwloss,bwacc],'after':[atloss,atacc,awloss,awacc]},index=["tloss","tacc","wloss","wacc"])
df.to_csv(rf'{folder_name}/res.csv')